# Home: render the film on Colab's free GPU

1. **Runtime → Change runtime type → T4 GPU**, then run the cells from top to bottom.
2. The frames go to your Google Drive in `MyDrive/home-film/renders/sceneNN/`. Frames that already exist are **skipped**, so if Colab disconnects, run everything again and it carries on where it stopped.
3. Change `SCENES` to choose which scenes to render this session. A free session lasts a few hours, so do a few scenes per day.

The song is never uploaded here. The final edit with music happens on your own computer (see the README).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/home-film'
import os; os.makedirs(DRIVE + '/renders', exist_ok=True)

In [ ]:
# Blender 5.3 for Linux (official build), and the film scripts from GitHub
!wget -q -nc https://download.blender.org/release/Blender5.3/blender-5.3.0-linux-x64.tar.xz -O /content/blender.tar.xz
!tar -xf /content/blender.tar.xz -C /content && ln -sf /content/blender-5.3.0-linux-x64/blender /usr/local/bin/blender
!rm -rf /content/Valdek && git clone -q --depth 1 -b claude/gallant-wozniak-uq8qof https://github.com/valdekluik5-ui/Valdek.git /content/Valdek
!blender --version | head -1

In [ ]:
# ---- what to render this session
SCENES  = ['01', '02', '03']          # e.g. ['04'] or all: [f'{i:02d}' for i in range(1, 17)]
ENGINE  = 'cycles'                     # 'cycles' (best on the T4 GPU) or 'eevee'
PERCENT = 67                           # 67% of 1080x1920 = 720x1280 (sharp on a phone). 100 = full HD
SAMPLES = 64                           # Cycles samples (with denoising). 32 = faster, 128 = cleaner
STEP    = 1                            # 1 = every frame (final). 2 = every second frame (quick preview)

In [ ]:
import glob, subprocess, time
SCRIPTS = {os.path.basename(p)[5:7]: p for p in glob.glob('/content/Valdek/home-film/scene[0-9][0-9]_*.py')}
for sid in SCENES:
    script = SCRIPTS[sid]
    out = f'{DRIVE}/renders/scene{sid}'
    src = open('/content/Valdek/home-film/film.py').read()
    # frame count from the timeline in film.py
    import re
    t0, t1 = map(float, re.search(r"'%s': dict\(name='[^']*', t=\(([\d.]+), ([\d.]+)\)" % sid, src).groups())
    n = int(round((t1 - t0) * 24))
    t = time.time()
    print(f'scene {sid}: {n} frames -> {out}')
    cmd = ['blender', '-b', '--python', script, '--', '--engine', ENGINE, '--percent', str(PERCENT), '--samples', str(SAMPLES),
           '--frames', f'1-{n}x{STEP}', '--out', out, '--nosave'] + (['--gpu'] if ENGINE == 'cycles' else [])
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if 'FRAME' in line or 'Error' in line or 'Traceback' in line or 'Cycles on' in line:
            print(line.rstrip())
    print(f'scene {sid} done in {(time.time() - t) / 60:.1f} min')

In [ ]:
# quick check: how many frames each scene has on Drive
for d in sorted(glob.glob(DRIVE + '/renders/scene*')):
    print(os.path.basename(d), len(glob.glob(d + '/*.png')), 'frames')